# Nonprofit density and community need by census tract (Massachusetts)

**Milestone 3 · enrichment view mockups.** Two maps, both built from the tract table made in the ZIP-fallback /
ACS step (`ma_tract_kpi1.csv`):

1. **Nonprofit density:** registered nonprofits per 1,000 residents in each census tract.
2. **Density vs. need:** each tract's nonprofit density compared with its poverty rate, to flag *possible*
   nonprofit deserts (high need, few nonprofits) and *possible* over-concentration (low need, many nonprofits).

The cut-offs in map 2 are a **proposal for the client to review**, not a final definition of a "gap in support".

Run top to bottom. Edit only the **Config** cell.

## Config

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Colab: True = upload ma_tract_kpi1.csv from your computer and download the results as a zip.
#        False = read from / save to Google Drive.
UPLOAD_FROM_COMPUTER = True

# Folder that holds task2_outputs/ma_tract_kpi1.csv (used when not uploading).
DATA_DIR = Path("/content/drive/MyDrive") if IN_COLAB else Path("data")
TRACT_FILE = "ma_tract_kpi1.csv"
SAVE_DIR_NAME = "nonprofit_density_maps"   # created next to the tract file's folder

# Map 2 settings
NEED_VAR = "poverty_rate_pct"       # or "median_household_income" (then low income = high need)
SPLIT = 1 / 3                       # top / bottom third of tracts; 0.5 = split at the median
STATE_FIPS = "25"
TOP_N = 15

## Setup

In [ ]:
import subprocess

if IN_COLAB:
    if UPLOAD_FROM_COMPUTER:
        from google.colab import files
        Path("/content/upload").mkdir(exist_ok=True)
        print("Choose ma_tract_kpi1.csv from your computer")
        for fname, data in files.upload().items():
            (Path("/content/upload") / fname).write_bytes(data)
        DATA_DIR = Path("/content/upload")
    else:
        from google.colab import drive
        drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas", "folium"], check=True)

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)


def locate(name: str) -> Path:
    p = DATA_DIR / name
    if p.exists():
        return p
    for root in [DATA_DIR] + ([Path("/content/drive/MyDrive")] if IN_COLAB and not UPLOAD_FROM_COMPUTER else []):
        if root.exists():
            print(f"  searching {root} for {name} ...")
            hit = next(root.rglob(name), None)
            if hit:
                return hit
    raise FileNotFoundError(f"Could not find {name}. Set DATA_DIR in the Config cell (currently {DATA_DIR}).")


TRACT_PATH = locate(TRACT_FILE)
if IN_COLAB and UPLOAD_FROM_COMPUTER:
    SAVE_DIR = Path("/content") / SAVE_DIR_NAME
else:
    SAVE_DIR = TRACT_PATH.parent.parent / SAVE_DIR_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Tract table:", TRACT_PATH)
print("Outputs go to", SAVE_DIR)

## 1. Load the tract table

One row per MA census tract: ACS indicators, the number of nonprofits placed in the tract (geocoded address, or
ZIP fallback for PO boxes and unmatched addresses) and `kpi1_nonprofits_per_1k`. Tracts under 500 residents are
flagged (`low_population_flag`): a single nonprofit there gives a very high rate, so they are shown in gray and
left out of the comparisons.

In [ ]:
t = pd.read_csv(TRACT_PATH, dtype={"geoid": str, "county_fips": str})
t["geoid"] = t.geoid.str.zfill(11)
t["low_population_flag"] = t.low_population_flag.astype(str).str.lower().isin(["true", "1"])
num = ["population", "median_household_income", "poverty_rate_pct", "nonprofit_count", "kpi1_nonprofits_per_1k"]
t[num] = t[num].apply(pd.to_numeric, errors="coerce")
stable = t[~t.low_population_flag & t.kpi1_nonprofits_per_1k.notna()].copy()

state_rate = 1000 * t.nonprofit_count.sum() / t.population.sum()
print(f"Tracts: {len(t):,} | low-population (gray): {t.low_population_flag.sum():,} | "
      f"zero nonprofits: {(t.nonprofit_count == 0).sum():,}")
print(f"Statewide: {t.nonprofit_count.sum():,} nonprofits, {t.population.sum():,.0f} residents, "
      f"{state_rate:.2f} per 1,000")
print(stable.kpi1_nonprofits_per_1k.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2).to_string())

## 2. Classify tracts for map 2 (density vs. need)

Among tracts with 500+ residents:

| Group | Rule |
|---|---|
| Possible nonprofit desert | need in the top third, density in the bottom third |
| Possible over-concentration | need in the bottom third, density in the top third |
| High need, many nonprofits | need in the top third, density in the top third |
| Other | everything else |

"Need" is the poverty rate by default. With `NEED_VAR = "median_household_income"`, lower income counts as higher
need. These are descriptive flags for discussion, not a judgment on any organization.

In [ ]:
need = stable[NEED_VAR]
if NEED_VAR == "median_household_income":
    need = -need                       # lower income = higher need
dens = stable.kpi1_nonprofits_per_1k
hi_need = need >= need.quantile(1 - SPLIT)
lo_need = need <= need.quantile(SPLIT)
hi_dens = dens >= dens.quantile(1 - SPLIT)
lo_dens = dens <= dens.quantile(SPLIT)
GROUPS = ["Possible nonprofit desert", "Possible over-concentration", "High need, many nonprofits", "Other"]
stable["group"] = np.select([hi_need & lo_dens, lo_need & hi_dens, hi_need & hi_dens], GROUPS[:3], GROUPS[3])
t = t.merge(stable[["geoid", "group"]], on="geoid", how="left")
t["group"] = t.group.fillna("Low population (excluded)")

cut = pd.DataFrame([
    {"measure": NEED_VAR, "top third from": round(stable[NEED_VAR].quantile(1 - SPLIT) if NEED_VAR != "median_household_income" else stable[NEED_VAR].quantile(SPLIT), 2),
     "bottom third up to": round(stable[NEED_VAR].quantile(SPLIT) if NEED_VAR != "median_household_income" else stable[NEED_VAR].quantile(1 - SPLIT), 2)},
    {"measure": "nonprofits per 1,000", "top third from": round(dens.quantile(1 - SPLIT), 2),
     "bottom third up to": round(dens.quantile(SPLIT), 2)},
])
print("Cut-offs (tracts with 500+ residents):")
print(cut.to_string(index=False))
grp = (t.groupby("group").agg(tracts=("geoid", "size"), residents=("population", "sum"),
                              nonprofits=("nonprofit_count", "sum"))
       .reindex(GROUPS + ["Low population (excluded)"]))
grp["share_of_residents_%"] = (100 * grp.residents / t.population.sum()).round(1)
print("\n", grp.to_string())

deserts = (t[t.group == GROUPS[0]]
           .sort_values([NEED_VAR, "kpi1_nonprofits_per_1k"], ascending=[NEED_VAR == "median_household_income", True])
           [["geoid", "tract_name", "population", "poverty_rate_pct", "median_household_income",
             "nonprofit_count", "kpi1_nonprofits_per_1k"]])
print(f"\nPossible nonprofit deserts, highest need first (top {TOP_N}):")
print(deserts.head(TOP_N).to_string(index=False))

## 3. Scatter: density vs. need

A quick check of the relationship before the maps. Each dot is a tract with 500+ residents.

In [ ]:
import matplotlib.pyplot as plt

COLORS = {GROUPS[0]: "#D9822B", GROUPS[1]: "#2F6690", GROUPS[2]: "#4CA394", GROUPS[3]: "#C9C9C9",
          "Low population (excluded)": "#EDEDED"}
fig, ax = plt.subplots(figsize=(11, 7))
for name in [GROUPS[3], GROUPS[2], GROUPS[1], GROUPS[0]]:
    d = stable[stable.group == name]
    ax.scatter(d[NEED_VAR], d.kpi1_nonprofits_per_1k, s=14, alpha=0.75, color=COLORS[name], label=f"{name} ({len(d):,})")
ax.set_yscale("symlog", linthresh=1)
ax.set_xlabel({"poverty_rate_pct": "Poverty rate (%)",
               "median_household_income": "Median household income ($)"}.get(NEED_VAR, NEED_VAR.replace("_", " ")))
ax.set_ylabel("Nonprofits per 1,000 residents (symlog scale)")
ax.set_title("Nonprofit density vs. need, Massachusetts census tracts")
ax.legend(frameon=False)
corr = stable[[NEED_VAR, "kpi1_nonprofits_per_1k"]].corr(method="spearman").iloc[0, 1]
ax.text(0.99, 0.02, f"Spearman correlation: {corr:.2f}", transform=ax.transAxes, ha="right", color="#555555")
fig.savefig(SAVE_DIR / "density_vs_need_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Maps

Tract boundaries are the Census cartographic boundary file for Massachusetts (2020 tract geography, which the
2020–2024 ACS uses).

In [ ]:
import io, zipfile
import requests
import geopandas as gpd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

CACHE = Path("/content/tract_cache") if IN_COLAB else Path("tract_cache")
CACHE.mkdir(parents=True, exist_ok=True)
tracts = None
for yr in [2024, 2023, 2022, 2020]:
    url = f"https://www2.census.gov/geo/tiger/GENZ{yr}/shp/cb_{yr}_{STATE_FIPS}_tract_500k.zip"
    folder = CACHE / Path(url).stem
    try:
        if not folder.exists():
            r = requests.get(url, timeout=300)
            r.raise_for_status()
            zipfile.ZipFile(io.BytesIO(r.content)).extractall(folder)
        tracts = gpd.read_file(next(folder.glob("*.shp")))[["GEOID", "geometry"]]
        print("Boundaries:", url)
        break
    except Exception as e:
        print("Failed:", url, e)
gt = tracts.merge(t, left_on="GEOID", right_on="geoid", how="left")
print(f"Tracts drawn: {len(gt):,}; tracts in the table but not in the boundary file: "
      f"{(~t.geoid.isin(tracts.GEOID)).sum():,}")

### Map 1: nonprofits per 1,000 residents

Colors are quintiles of tracts with 500+ residents, so a few very high downtown tracts don't wash out the rest.

In [ ]:
bins = list(np.unique(np.quantile(stable.kpi1_nonprofits_per_1k, [0, .2, .4, .6, .8, 1])))
bins[-1] = max(bins[-1], gt.kpi1_nonprofits_per_1k.max() or 0) + 1e-9
cmap = ListedColormap(["#EAF4F1", "#B9DDD3", "#7FC0AF", "#4CA394", "#1F6F61"][:len(bins) - 1])
norm = BoundaryNorm(bins, cmap.N)
fig, ax = plt.subplots(figsize=(16, 10))
show = gt[~gt.low_population_flag.fillna(True)]
gt[gt.low_population_flag.fillna(True)].plot(ax=ax, color="#EDEDED", edgecolor="#FFFFFF", linewidth=0.1)
show.plot(ax=ax, column="kpi1_nonprofits_per_1k", cmap=cmap, norm=norm, edgecolor="#FFFFFF", linewidth=0.1)
labels = [f"{bins[i]:.1f}–{bins[i + 1]:.1f}" if i < len(bins) - 2 else f"{bins[i]:.1f}+" for i in range(len(bins) - 1)]
handles = [Patch(color=cmap(i), label=l) for i, l in enumerate(labels)] + [Patch(color="#EDEDED", label="Under 500 residents")]
ax.legend(handles=handles, title="Nonprofits per 1,000 residents", loc="lower left", frameon=False)
ax.set_axis_off()
ax.set_title(f"Registered nonprofits per 1,000 residents by census tract, Massachusetts\n"
             f"Statewide: {state_rate:.2f} per 1,000", fontsize=16)
fig.text(0.12, 0.06, "Nonprofits are counted at their filing address (office or mailing location), "
         "not their service area.", fontsize=10, color="#555555")
fig.savefig(SAVE_DIR / "map1_nonprofit_density.png", dpi=200, bbox_inches="tight")
plt.show()

### Map 2: possible nonprofit deserts vs. over-concentration

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
for name, color in COLORS.items():
    d = gt[gt.group.fillna("Low population (excluded)") == name]
    if len(d):
        d.plot(ax=ax, color=color, edgecolor="#FFFFFF", linewidth=0.1)
# Count from the tract table (same numbers as group_summary.csv), not from the drawn shapes
counts = t.group.value_counts()
handles = [Patch(color=c, label=f"{n} ({counts.get(n, 0):,} tracts)") for n, c in COLORS.items()]
ax.legend(handles=handles, loc="lower left", frameon=False)
ax.set_axis_off()
need_label = "poverty rate" if NEED_VAR == "poverty_rate_pct" else "median household income"
ax.set_title(f"Nonprofit density vs. need ({need_label}), Massachusetts census tracts", fontsize=16)
fig.text(0.12, 0.06, f"Desert = top third for need and bottom third for nonprofits per 1,000; over-concentration = "
         f"the reverse. Proposed cut-offs for client review.", fontsize=10, color="#555555")
fig.savefig(SAVE_DIR / "map2_density_vs_need.png", dpi=200, bbox_inches="tight")
plt.show()

### Interactive version of map 2 (hover a tract)

In [ ]:
try:
    import folium
    fm = gt.to_crs(4326).copy()
    fm["group"] = fm.group.fillna("Low population (excluded)")
    for c in ["population", "nonprofit_count"]:
        fm[c] = fm[c].fillna(0).astype(int)
    fm["per_1k"] = fm.kpi1_nonprofits_per_1k.round(2)
    fm["poverty"] = fm.poverty_rate_pct.round(1)
    fm["income"] = fm.median_household_income.map(lambda v: "" if pd.isna(v) else f"${v:,.0f}")
    mobj = folium.Map(location=[42.25, -71.8], zoom_start=8, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
                      attr="Tiles &copy; Esri &mdash; Esri, HERE, Garmin, OpenStreetMap contributors", max_zoom=16)
    folium.GeoJson(
        fm[["GEOID", "tract_name", "group", "population", "nonprofit_count", "per_1k", "poverty", "income",
            "geometry"]].__geo_interface__,
        style_function=lambda f: {"fillColor": COLORS.get(f["properties"]["group"], "#EDEDED"),
                                  "color": "#FFFFFF", "weight": 0.3, "fillOpacity": 0.8},
        tooltip=folium.GeoJsonTooltip(
            fields=["tract_name", "group", "population", "nonprofit_count", "per_1k", "poverty", "income"],
            aliases=["Tract", "Group", "Residents", "Nonprofits", "Per 1,000", "Poverty %", "Median income"]),
    ).add_to(mobj)
    mobj.save(str(SAVE_DIR / "map2_density_vs_need.html"))
    display(mobj)
except ImportError:
    print("folium not installed; skipping the interactive map")

## 5. Save

In [ ]:
t.to_csv(SAVE_DIR / "tracts_with_groups.csv", index=False)
grp.to_csv(SAVE_DIR / "group_summary.csv")
cut.to_csv(SAVE_DIR / "cutoffs.csv", index=False)
deserts.to_csv(SAVE_DIR / "possible_deserts.csv", index=False)
for p in sorted(SAVE_DIR.iterdir()):
    print(f"{p.name:35s} {p.stat().st_size / 1e3:8.1f} KB")

# Download everything to your computer as one zip
if IN_COLAB and UPLOAD_FROM_COMPUTER:
    import shutil
    from google.colab import files
    zip_path = shutil.make_archive(f"/content/{SAVE_DIR_NAME}", "zip", SAVE_DIR)
    files.download(zip_path)